# Punto 3 — Clasificación de lengua de señas (27 clases)

**Dataset:** [27 Class Sign Language Dataset — ardamavi](https://www.kaggle.com/datasets/ardamavi/27-class-sign-language-dataset)

**Objetivo:** Entrenar redes neuronales densas que clasifiquen imágenes de mano en 27 clases de lengua de señas basada en ASL.

**Modelos:**
1. DNN Base — 1 capa oculta grande
2. DNN con regularización — Dropout + BatchNormalization
3. DNN Profunda — 4 capas con unidades decrecientes

**Preprocesamiento:** Las imágenes se aplanan a vectores 1-D para ser compatibles con capas Dense.

## 1. Importaciones

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, confusion_matrix, classification_report
)
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.utils import to_categorical

tf.random.set_seed(42)
np.random.seed(42)

sns.set_theme(style='whitegrid')
DATA_DIR = Path('../data/sign_language')

print(f'TensorFlow {tf.__version__}')

## 2. Descarga del dataset

Requiere tener la [API de Kaggle](https://www.kaggle.com/docs/api) configurada (`~/.kaggle/kaggle.json`).
Si ya descargaste el dataset manualmente, coloca los archivos en `../data/sign_language/`.

In [ ]:
DATA_DIR.mkdir(parents=True, exist_ok=True)

npy_files = list(DATA_DIR.glob('*.npy'))

if not npy_files:
    print('Descargando dataset desde Kaggle...')
    os.system(f'kaggle datasets download -d ardamavi/27-class-sign-language-dataset -p {DATA_DIR} --unzip')
    npy_files = list(DATA_DIR.glob('**/*.npy'))

print('Archivos .npy encontrados:')
for f in npy_files:
    print(f'  {f.name}  —  {f.stat().st_size / 1e6:.1f} MB')

## 3. Carga y exploración de datos

In [ ]:
# El dataset contiene X.npy (imágenes) e Y.npy (labels one-hot o enteros)
X_path = next(DATA_DIR.glob('**/X.npy'), None) or next(DATA_DIR.glob('**/*data*.npy'), None)
Y_path = next(DATA_DIR.glob('**/Y.npy'), None) or next(DATA_DIR.glob('**/*label*.npy'), None)

# Si los archivos tienen nombres distintos, ajustar aquí:
if X_path is None or Y_path is None:
    all_npy = sorted(DATA_DIR.glob('**/*.npy'))
    print('Archivos disponibles:',  [f.name for f in all_npy])
    # Asignar manualmente si es necesario
    X_path, Y_path = all_npy[0], all_npy[1]

X = np.load(X_path)
Y = np.load(Y_path)

print(f'X shape: {X.shape}  dtype: {X.dtype}')
print(f'Y shape: {Y.shape}  dtype: {Y.dtype}')

In [ ]:
# Normalizar etiquetas: si Y es one-hot, convertir a enteros
if Y.ndim == 2:
    y_int = np.argmax(Y, axis=1)
else:
    y_int = Y.astype(int)

NUM_CLASSES = len(np.unique(y_int))
print(f'Número de clases: {NUM_CLASSES}')
print(f'Distribución: {np.bincount(y_int)}')

In [ ]:
# Normalizar píxeles y aplanar
X_norm = X.astype(np.float32) / 255.0

# Si las imágenes tienen 4 dimensiones (N, H, W, C), aplanar
if X_norm.ndim == 4:
    N, H, W, C = X_norm.shape
    X_flat = X_norm.reshape(N, H * W * C)
elif X_norm.ndim == 3:
    N, H, W = X_norm.shape
    C = 1
    X_flat = X_norm.reshape(N, H * W)
else:
    X_flat = X_norm  # ya es 2D

print(f'X_flat shape: {X_flat.shape}  (N={X_flat.shape[0]}, features={X_flat.shape[1]})')

In [ ]:
# Visualizar algunas muestras
fig, axes = plt.subplots(3, 9, figsize=(18, 6))
for cls in range(min(27, NUM_CLASSES)):
    idx = np.where(y_int == cls)[0][0]
    ax  = axes[cls // 9][cls % 9]
    img = X[idx]
    if img.ndim == 3:
        ax.imshow(img.astype(np.uint8))
    else:
        ax.imshow(img, cmap='gray')
    ax.set_title(f'Clase {cls}', fontsize=7)
    ax.axis('off')

plt.suptitle('Una muestra por clase del dataset de lengua de señas', fontsize=12)
plt.tight_layout()
plt.show()

## 4. División train/test

In [ ]:
X_train, X_test, y_train_int, y_test_int = train_test_split(
    X_flat, y_int,
    test_size=0.2,
    random_state=42,
    stratify=y_int
)

y_train = to_categorical(y_train_int, num_classes=NUM_CLASSES)
y_test  = to_categorical(y_test_int,  num_classes=NUM_CLASSES)

INPUT_DIM = X_train.shape[1]
print(f'Train: {X_train.shape}  Test: {X_test.shape}')
print(f'Input dimension: {INPUT_DIM}')

## 5. Construcción de modelos

In [ ]:
def build_dnn_base(input_dim, num_classes):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(256, activation='relu'),
        layers.Dense(num_classes, activation='softmax')
    ], name='DNN_Base')
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model


def build_dnn_regularized(input_dim, num_classes):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.4),
        layers.Dense(128, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation='softmax')
    ], name='DNN_Regularized')
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model


def build_dnn_deep(input_dim, num_classes):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(512, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.4),
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(64, activation='relu'),
        layers.Dense(num_classes, activation='softmax')
    ], name='DNN_Deep')
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model


MODELS = {
    'DNN Base':        build_dnn_base,
    'DNN Regularized': build_dnn_regularized,
    'DNN Deep':        build_dnn_deep,
}

EPOCHS     = 80
BATCH_SIZE = 64

## 6. Entrenamiento

In [ ]:
trained_models  = {}
histories       = {}
results_sl      = []

for name, builder in MODELS.items():
    print(f'\n{"="*55}')
    print(f'  Entrenando: {name}')
    print(f'{"="*55}')

    tf.random.set_seed(42)
    model = builder(INPUT_DIM, NUM_CLASSES)

    callbacks = [
        keras.callbacks.EarlyStopping(monitor='val_loss', patience=12, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=6, verbose=0)
    ]

    hist = model.fit(
        X_train, y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_split=0.15,
        callbacks=callbacks,
        verbose=1
    )

    trained_models[name] = model
    histories[name] = hist

    y_pred_prob = model.predict(X_test, verbose=0)
    y_pred_int  = np.argmax(y_pred_prob, axis=1)

    acc  = accuracy_score(y_test_int, y_pred_int)
    f1   = f1_score(y_test_int, y_pred_int, average='macro')
    f1w  = f1_score(y_test_int, y_pred_int, average='weighted')

    print(f'  Accuracy: {acc:.4f}  |  F1 macro: {f1:.4f}  |  F1 weighted: {f1w:.4f}')
    results_sl.append({
        'Modelo': name,
        'Accuracy': acc,
        'F1 Macro': f1,
        'F1 Weighted': f1w
    })

print('\n✓ Entrenamiento completo.')

## 7. Tabla comparativa de resultados

In [ ]:
df_sl = pd.DataFrame(results_sl).set_index('Modelo')
print('=== Resultados — Clasificación de Lengua de Señas ===')
display(df_sl.round(4))

## 8. Curvas de entrenamiento

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for col, (name, hist) in enumerate(histories.items()):
    axes[0, col].plot(hist.history['loss'],     label='Train')
    axes[0, col].plot(hist.history['val_loss'], label='Validación')
    axes[0, col].set_title(f'{name} — Loss')
    axes[0, col].set_xlabel('Época')
    axes[0, col].set_ylabel('Categorical Cross-Entropy')
    axes[0, col].legend()

    axes[1, col].plot(hist.history['accuracy'],     label='Train')
    axes[1, col].plot(hist.history['val_accuracy'], label='Validación')
    axes[1, col].set_title(f'{name} — Accuracy')
    axes[1, col].set_xlabel('Época')
    axes[1, col].set_ylabel('Accuracy')
    axes[1, col].legend()

plt.suptitle('Curvas de entrenamiento — Lengua de señas', fontsize=13)
plt.tight_layout()
plt.show()

## 9. Matriz de confusión (mejor modelo)

In [ ]:
# Usar el modelo con mayor F1 macro
best_name = df_sl['F1 Macro'].idxmax()
best_model = trained_models[best_name]

y_pred_best = np.argmax(best_model.predict(X_test, verbose=0), axis=1)
cm = confusion_matrix(y_test_int, y_pred_best)

plt.figure(figsize=(16, 14))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=range(NUM_CLASSES),
            yticklabels=range(NUM_CLASSES))
plt.title(f'Matriz de confusión — {best_name}', fontsize=13)
plt.xlabel('Clase predicha')
plt.ylabel('Clase real')
plt.tight_layout()
plt.show()

print(f'\nMejor modelo: {best_name}')
print(classification_report(y_test_int, y_pred_best))

## 10. Análisis por clase — F1-score

In [ ]:
f1_per_class = f1_score(y_test_int, y_pred_best, average=None)

plt.figure(figsize=(14, 4))
colors = ['tomato' if f < 0.7 else 'steelblue' for f in f1_per_class]
plt.bar(range(NUM_CLASSES), f1_per_class, color=colors)
plt.axhline(y=np.mean(f1_per_class), color='black', linestyle='--', label=f'Promedio = {np.mean(f1_per_class):.3f}')
plt.title(f'F1-score por clase — {best_name}')
plt.xlabel('Clase')
plt.ylabel('F1-score')
plt.legend()
plt.xticks(range(NUM_CLASSES))
plt.tight_layout()
plt.show()

worst = np.argsort(f1_per_class)[:5]
best_classes = np.argsort(f1_per_class)[-5:]
print(f'Clases con peor F1:  {worst.tolist()} → {f1_per_class[worst].round(3).tolist()}')
print(f'Clases con mejor F1: {best_classes.tolist()} → {f1_per_class[best_classes].round(3).tolist()}')

## 11. Análisis de resultados

### Observaciones principales

1. **Aplanamiento de imágenes (Flatten):** Al usar capas Dense se pierde la estructura espacial 2D de la imagen. Esto limita el rendimiento en comparación con CNNs, pero el ejercicio exige usar redes densas. Con imágenes de tamaño moderado (ej. 64×64×3 = 12288 features) la DNN todavía puede aprender patrones.

2. **DNN Base vs. Regularizada:** La DNN Base puede sobreajustar rápidamente porque tiene muchos parámetros sin restricciones. BatchNormalization estabiliza el entrenamiento y Dropout reduce la co-adaptación de neuronas.

3. **DNN Profunda:** Más capas permiten aprender representaciones jerárquicas, pero el gradiente puede disiparse. El uso combinado de `ReduceLROnPlateau` y `EarlyStopping` ayuda a encontrar el mínimo correcto.

4. **Clases difíciles:** Las matrices de confusión revelan qué señas se confunden entre sí. Señas visualmente similares (dedos en posiciones parecidas) tendrán mayor confusión. Esto es una limitación intrínseca de usar solo imágenes sin información de profundidad.

5. **Métrica elegida — F1 macro:** En clasificación multiclase desbalanceada, la F1 macro trata todas las clases por igual (útil para detectar clases con poco soporte). La F1 weighted pondera por frecuencia.

6. **Limitación:** Las DNNs densas son subóptimas para imágenes. En un escenario real se usarían CNNs o transfer learning, que logran >98% de accuracy en este tipo de dataset.